# EdTech Learning Outcome Analytics

## Day 4 — Feature Engineering

### Objective

The objective of this notebook is to transform the cleaned OULAD datasets into analytical features at the student-course-presentation level.

The features will cover:

- Student characteristics
- Registration behavior
- VLE engagement
- Assessment performance
- Early learning engagement
- Course characteristics

The resulting Master Analytical Dataset will be used for SQL analysis, Power BI reporting, and dropout-risk modeling.

### Analytical Grain

One row represents:

Student + Course + Course Presentation

In [11]:
import pandas as pd
import numpy as np
import os

In [9]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

In [10]:
processed_path = "../data/processed"

print("Files in processed folder:")
for file in sorted(os.listdir(processed_path)):
    print(file)

Files in processed folder:
assessments_cleaned.csv
student_assessment_cleaned.csv
student_info_cleaned.csv
student_registration_cleaned.csv
student_vle_cleaned.csv
vle_cleaned.csv


In [12]:
processed_path = "../data/processed"

student_info = pd.read_csv(
    os.path.join(processed_path, "student_info_cleaned.csv")
)

student_registration = pd.read_csv(
    os.path.join(processed_path, "student_registration_cleaned.csv")
)

student_vle = pd.read_csv(
    os.path.join(processed_path, "student_vle_cleaned.csv")
)

vle = pd.read_csv(
    os.path.join(processed_path, "vle_cleaned.csv")
)

assessments = pd.read_csv(
    os.path.join(processed_path, "assessments_cleaned.csv")
)

student_assessment = pd.read_csv(
    os.path.join(processed_path, "student_assessment_cleaned.csv")
)

In [13]:
datasets = {
    "Student Info": student_info,
    "Registration": student_registration,
    "VLE Activity": student_vle,
    "VLE Metadata": vle,
    "Assessments": assessments,
    "Student Assessment": student_assessment
}

print("=== DATASET VALIDATION ===")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

=== DATASET VALIDATION ===
Student Info: 32,593 rows × 12 columns
Registration: 32,593 rows × 5 columns
VLE Activity: 9,868,110 rows × 6 columns
VLE Metadata: 6,364 rows × 6 columns
Assessments: 206 rows × 6 columns
Student Assessment: 173,912 rows × 5 columns


In [14]:
student_info.duplicated(
    subset=[
        "id_student",
        "code_module",
        "code_presentation"
    ]
).sum()

np.int64(0)

In [15]:
master_df = student_info.copy()

In [16]:
print("Master dataset rows:", len(master_df))
print("Master dataset columns:", len(master_df.columns))

Master dataset rows: 32593
Master dataset columns: 12


In [17]:
master_df.columns.tolist()

['code_module',
 'code_presentation',
 'id_student',
 'gender',
 'region',
 'highest_education',
 'imd_band',
 'age_band',
 'num_of_prev_attempts',
 'studied_credits',
 'disability',
 'final_result']

## Step 9 — Create Dropout Target

The `dropout` target identifies whether a student-course-presentation record ended with a `Withdrawn` outcome.

- 1 = Withdrawn
- 0 = All other final outcomes

In [18]:
master_df["dropout"] = (
    master_df["final_result"] == "Withdrawn"
).astype("int8")

In [19]:
print(master_df["dropout"].value_counts())
print()
print(master_df["dropout"].value_counts(normalize=True).mul(100).round(2))

dropout
0    22437
1    10156
Name: count, dtype: int64

dropout
0    68.84
1    31.16
Name: proportion, dtype: float64


In [20]:
print("Master rows:", len(master_df))
print("Master columns:", len(master_df.columns))

Master rows: 32593
Master columns: 13


In [21]:
student_registration.columns.tolist()

['code_module',
 'code_presentation',
 'id_student',
 'date_registration',
 'date_unregistration']

In [22]:
student_registration.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


In [23]:
student_registration.duplicated(
    subset=[
        "id_student",
        "code_module",
        "code_presentation"
    ]
).sum()

np.int64(0)

In [24]:
master_df = master_df.merge(
    student_registration[
        [
            "id_student",
            "code_module",
            "code_presentation",
            "date_registration",
            "date_unregistration"
        ]
    ],
    on=[
        "id_student",
        "code_module",
        "code_presentation"
    ],
    how="left",
    validate="one_to_one"
)

In [25]:
print("Rows after registration merge:", len(master_df))
print("Columns after registration merge:", len(master_df.columns))


Rows after registration merge: 32593
Columns after registration merge: 15


In [26]:
master_df[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date_registration",
        "date_unregistration"
    ]
].head()

,id_student,code_module,code_presentation,date_registration,date_unregistration
0,11391,AAA,2013J,-159.0,NaN
1,28400,AAA,2013J,-53.0,NaN
2,30268,AAA,2013J,-92.0,12.0
3,31604,AAA,2013J,-52.0,NaN
4,32885,AAA,2013J,-176.0,NaN


In [27]:
master_df[
    ["date_registration", "date_unregistration"]
].isnull().sum()

date_registration         45
date_unregistration    22521
dtype: int64

In [28]:
master_df["registration_missing"] = (
    master_df["date_registration"].isna()
).astype("int8")

In [29]:
master_df["was_unregistered"] = (
    master_df["date_unregistration"].notna()
).astype("int8")

In [30]:
master_df["days_until_unregistration"] = (
    master_df["date_unregistration"] -
    master_df["date_registration"]
)

In [31]:
master_df["days_until_unregistration"].describe()

count    10033.000000
mean       128.797568
std         85.694853
min          0.000000
25%         57.000000
50%        117.000000
75%        186.000000
max        531.000000
Name: days_until_unregistration, dtype: float64

In [32]:
master_df[
    "days_until_unregistration"
].isna().sum()

np.int64(22560)

In [33]:
print(
    master_df.duplicated(
        subset=[
            "id_student",
            "code_module",
            "code_presentation"
        ]
    ).sum()
)

0


In [34]:
print("Rows:", len(master_df))

Rows: 32593


In [35]:
master_df["dropout"].value_counts(dropna=False)

dropout
0    22437
1    10156
Name: count, dtype: int64

In [36]:
master_df.shape

(32593, 18)

In [37]:
master_df[
    [
        "date_registration",
        "date_unregistration",
        "registration_missing",
        "was_unregistered",
        "days_until_unregistration"
    ]
].head()

,date_registration,date_unregistration,registration_missing,was_unregistered,days_until_unregistration
0,-159.0,NaN,0,0,NaN
1,-53.0,NaN,0,0,NaN
2,-92.0,12.0,0,1,104.0
3,-52.0,NaN,0,0,NaN
4,-176.0,NaN,0,0,NaN


In [38]:
master_df["days_until_unregistration"].describe()

count    10033.000000
mean       128.797568
std         85.694853
min          0.000000
25%         57.000000
50%        117.000000
75%        186.000000
max        531.000000
Name: days_until_unregistration, dtype: float64

In [39]:
master_df["dropout"].value_counts()

dropout
0    22437
1    10156
Name: count, dtype: int64

## Step 16 — Full-Course VLE Engagement Features

The Student VLE dataset contains individual learning-interaction records.

To align the data with the analytical grain of the Master Dataset, VLE activity will be aggregated at:

Student + Course + Course Presentation

The following engagement features will be created:

- Total VLE Clicks
- Active Learning Days
- Average Clicks per Active Day
- Number of Active VLE Days as a proportion of course duration

In [40]:
vle_engagement = (
    student_vle
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation"
        ]
    )
    .agg(
        total_clicks=("sum_click", "sum"),
        active_days=("date", "nunique"),
        first_activity_date=("date", "min"),
        last_activity_date=("date", "max")
    )
    .reset_index()
)

In [41]:
vle_engagement["activity_span_days"] = (
    vle_engagement["last_activity_date"]
    - vle_engagement["first_activity_date"]
)

In [42]:
vle_engagement.head()

,id_student,code_module,code_presentation,total_clicks,active_days,first_activity_date,last_activity_date,activity_span_days
0,6516,AAA,2014J,2715,159,-23,269,292
1,8462,DDD,2013J,634,56,-6,118,124
2,8462,DDD,2014J,9,1,10,10,0
3,11391,AAA,2013J,922,40,-5,253,258
4,23629,BBB,2013B,153,16,-6,87,93


In [43]:
vle_engagement["avg_clicks_per_active_day"] = (
    vle_engagement["total_clicks"] /
    vle_engagement["active_days"]
)

In [44]:
vle_engagement[
    [
        "total_clicks",
        "active_days",
        "avg_clicks_per_active_day"
    ]
].describe()

,total_clicks,active_days,avg_clicks_per_active_day
count,29228.000000,29228.000000,29228.000000
mean,1311.860647,61.862563,18.394078
std,1689.702260,54.027412,11.395633
min,1.000000,1.000000,1.000000
25%,247.750000,18.000000,10.659894
50%,709.000000,47.000000,15.734591
75%,1714.000000,92.000000,23.661352
max,23513.000000,286.000000,218.000000


In [47]:
vle_engagement.duplicated(
    subset=[
        "id_student",
        "code_module",
        "code_presentation"
    ]
).sum()

np.int64(0)

In [51]:
courses = pd.read_csv(
    os.path.join(
        "../data/raw",
        "courses.csv"
    )
)

In [52]:
print("Courses shape:", courses.shape)
courses.head()

Courses shape: (22, 3)


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240


In [53]:
course_duration = courses[
    [
        "code_module",
        "code_presentation",
        "module_presentation_length"
    ]
].copy()

In [54]:
course_duration.duplicated(
    subset=[
        "code_module",
        "code_presentation"
    ]
).sum()

np.int64(0)

## Step 17 — Add Course Duration to Engagement Features

Course presentation duration is added to the learner engagement dataset so that engagement can be normalized relative to the length of the course.

The merge is performed using `code_module` and `code_presentation`.

In [55]:
vle_engagement = vle_engagement.merge(
    course_duration,
    on=[
        "code_module",
        "code_presentation"
    ],
    how="left",
    validate="many_to_one"
)

In [56]:
print("Rows after course duration merge:", len(vle_engagement))
print("Columns:", len(vle_engagement.columns))

Rows after course duration merge: 29228
Columns: 10


In [57]:
vle_engagement[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "total_clicks",
        "active_days",
        "module_presentation_length"
    ]
].head()

,id_student,code_module,code_presentation,total_clicks,active_days,module_presentation_length
0,6516,AAA,2014J,2715,159,269
1,8462,DDD,2013J,634,56,261
2,8462,DDD,2014J,9,1,262
3,11391,AAA,2013J,922,40,268
4,23629,BBB,2013B,153,16,240


In [58]:
vle_engagement["module_presentation_length"].isnull().sum()

np.int64(0)

## Step 18 — Engagement Coverage

Engagement coverage measures the proportion of the course duration during which a learner was active.

Formula:

Engagement Coverage =
Active Learning Days / Course Presentation Length

In [59]:
vle_engagement["engagement_coverage"] = (
    vle_engagement["active_days"] /
    vle_engagement["module_presentation_length"]
)

In [61]:
vle_engagement[
    [
        "active_days",
        "module_presentation_length",
        "engagement_coverage"
    ]
].head()

,active_days,module_presentation_length,engagement_coverage
0,159,269,0.591078
1,56,261,0.214559
2,1,262,0.003817
3,40,268,0.149254
4,16,240,0.066667


In [62]:
vle_engagement["engagement_coverage"].describe()

count    29228.000000
mean         0.241718
std          0.211068
min          0.003717
25%          0.070632
50%          0.183761
75%          0.358779
max          1.072797
Name: engagement_coverage, dtype: float64

In [63]:
vle_engagement[
    (vle_engagement["engagement_coverage"] < 0) |
    (vle_engagement["engagement_coverage"] > 1)
].shape

(57, 11)


## Step 29 — Early Engagement Features

Early engagement features measure learner activity during the first 30 days of the course.

These features are particularly important for the later early-warning dropout prediction model because they use information available near the beginning of the learner journey.

In [64]:
early_vle = student_vle[
    (student_vle["date"] >= 0) &
    (student_vle["date"] <= 30)
].copy()

In [65]:
early_engagement = (
    early_vle
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation"
        ]
    )
    .agg(
        clicks_first_30_days=("sum_click", "sum"),
        active_days_first_30=("date", "nunique")
    )
    .reset_index()
)

In [66]:
early_engagement.head()

,id_student,code_module,code_presentation,clicks_first_30_days,active_days_first_30
0,6516,AAA,2014J,571,21
1,8462,DDD,2013J,325,17
2,8462,DDD,2014J,9,1
3,11391,AAA,2013J,326,9
4,23629,BBB,2013B,37,5


In [67]:
early_engagement["avg_clicks_first_30_active_day"] = np.where(
    early_engagement["active_days_first_30"] > 0,
    early_engagement["clicks_first_30_days"] /
    early_engagement["active_days_first_30"],
    0
)

In [68]:
early_engagement[
    [
        "clicks_first_30_days",
        "active_days_first_30",
        "avg_clicks_first_30_active_day"
    ]
].describe()

,clicks_first_30_days,active_days_first_30,avg_clicks_first_30_active_day
count,27971.000000,27971.000000,27971.000000
mean,291.610847,11.858282,21.254310
std,345.904371,7.709422,15.063573
min,1.000000,1.000000,1.000000
25%,74.000000,6.000000,11.066667
50%,184.000000,11.000000,17.875000
75%,388.000000,17.000000,27.707108
max,6495.000000,31.000000,260.368421


In [69]:
vle_features = vle_engagement.merge(
    early_engagement,
    on=[
        "id_student",
        "code_module",
        "code_presentation"
    ],
    how="left",
    validate="one_to_one"
)

In [70]:
early_columns = [
    "clicks_first_30_days",
    "active_days_first_30",
    "avg_clicks_first_30_active_day"
]

vle_features[early_columns] = (
    vle_features[early_columns]
    .fillna(0)
)

In [71]:
vle_features[early_columns].isnull().sum()

clicks_first_30_days              0
active_days_first_30              0
avg_clicks_first_30_active_day    0
dtype: int64

In [72]:
print("Rows:", len(vle_features))

Rows: 29228


In [73]:
print("Columns:", len(vle_features.columns))

Columns: 14


In [74]:
print(
    "Duplicate student-course-presentation records:",
    vle_features.duplicated(
        subset=[
            "id_student",
            "code_module",
            "code_presentation"
        ]
    ).sum()
)

Duplicate student-course-presentation records: 0


In [75]:
vle_features.isnull().sum()

id_student                        0
code_module                       0
code_presentation                 0
total_clicks                      0
active_days                       0
first_activity_date               0
last_activity_date                0
activity_span_days                0
avg_clicks_per_active_day         0
module_presentation_length        0
engagement_coverage               0
clicks_first_30_days              0
active_days_first_30              0
avg_clicks_first_30_active_day    0
dtype: int64

In [76]:
vle_features.head()

,id_student,code_module,code_presentation,total_clicks,active_days,first_activity_date,last_activity_date,activity_span_days,avg_clicks_per_active_day,module_presentation_length,engagement_coverage,clicks_first_30_days,active_days_first_30,avg_clicks_first_30_active_day
0,6516,AAA,2014J,2715,159,-23,269,292,17.075472,269,0.591078,571.0,21.0,27.190476
1,8462,DDD,2013J,634,56,-6,118,124,11.321429,261,0.214559,325.0,17.0,19.117647
2,8462,DDD,2014J,9,1,10,10,0,9.000000,262,0.003817,9.0,1.0,9.000000
3,11391,AAA,2013J,922,40,-5,253,258,23.050000,268,0.149254,326.0,9.0,36.222222
4,23629,BBB,2013B,153,16,-6,87,93,9.562500,240,0.066667,37.0,5.0,7.400000


In [77]:
vle_engagement.shape

(29228, 11)

In [78]:
vle_engagement["module_presentation_length"].isnull().sum()

np.int64(0)

In [79]:
vle_engagement["engagement_coverage"].describe()

count    29228.000000
mean         0.241718
std          0.211068
min          0.003717
25%          0.070632
50%          0.183761
75%          0.358779
max          1.072797
Name: engagement_coverage, dtype: float64

In [80]:
vle_features.shape

(29228, 14)

In [81]:
vle_features.isnull().sum()

id_student                        0
code_module                       0
code_presentation                 0
total_clicks                      0
active_days                       0
first_activity_date               0
last_activity_date                0
activity_span_days                0
avg_clicks_per_active_day         0
module_presentation_length        0
engagement_coverage               0
clicks_first_30_days              0
active_days_first_30              0
avg_clicks_first_30_active_day    0
dtype: int64

In [82]:
course_duration_map = course_duration.set_index(
    ["code_module", "code_presentation"]
)["module_presentation_length"]

In [84]:
student_vle_in_course = student_vle.copy()

In [85]:
student_vle_in_course = student_vle_in_course.merge(
    course_duration,
    on=["code_module", "code_presentation"],
    how="left",
    validate="many_to_one"
)

In [86]:
student_vle_in_course = student_vle_in_course[
    (student_vle_in_course["date"] >= 0) &
    (student_vle_in_course["date"] < student_vle_in_course["module_presentation_length"])
]

In [87]:
active_days_in_course = (
    student_vle_in_course
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation"
        ]
    )["date"]
    .nunique()
    .reset_index(name="active_days_in_course")
)

In [88]:
vle_engagement = vle_engagement.merge(
    active_days_in_course,
    on=[
        "id_student",
        "code_module",
        "code_presentation"
    ],
    how="left",
    validate="one_to_one"
)

vle_engagement["active_days_in_course"] = (
    vle_engagement["active_days_in_course"].fillna(0)
)

In [89]:
vle_engagement["engagement_coverage"] = (
    vle_engagement["active_days_in_course"] /
    vle_engagement["module_presentation_length"]
)

In [90]:
vle_engagement["engagement_coverage"].describe()

count    29228.000000
mean         0.227038
std          0.202059
min          0.000000
25%          0.062241
50%          0.171756
75%          0.340436
max          1.000000
Name: engagement_coverage, dtype: float64

In [91]:
vle_engagement[
    (vle_engagement["engagement_coverage"] < 0) |
    (vle_engagement["engagement_coverage"] > 1)
].shape

(0, 12)

In [92]:
vle_engagement["engagement_coverage"].describe()

count    29228.000000
mean         0.227038
std          0.202059
min          0.000000
25%          0.062241
50%          0.171756
75%          0.340436
max          1.000000
Name: engagement_coverage, dtype: float64

In [93]:
vle_engagement[
    (vle_engagement["engagement_coverage"] < 0) |
    (vle_engagement["engagement_coverage"] > 1)
].shape

(0, 12)

In [94]:
vle_engagement[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "active_days",
        "active_days_in_course",
        "module_presentation_length",
        "engagement_coverage"
    ]
].head()

,id_student,code_module,code_presentation,active_days,active_days_in_course,module_presentation_length,engagement_coverage
0,6516,AAA,2014J,159,149.0,269,0.553903
1,8462,DDD,2013J,56,51.0,261,0.195402
2,8462,DDD,2014J,1,1.0,262,0.003817
3,11391,AAA,2013J,40,39.0,268,0.145522
4,23629,BBB,2013B,16,15.0,240,0.062500


## Step 20 — Early Engagement Features

Early engagement features measure student activity during the first 30 days of a course presentation.

These features are intended for early-warning analysis and will later be used to identify whether early learner behaviour is associated with withdrawal risk.

Features created:

- Clicks in first 30 days
- Active days in first 30 days
- Average clicks per active day during first 30 days

In [95]:
early_vle = student_vle[
    (student_vle["date"] >= 0) &
    (student_vle["date"] <= 30)
].copy()

In [96]:
print("Early VLE rows:", len(early_vle))
print("Minimum date:", early_vle["date"].min())
print("Maximum date:", early_vle["date"].max())

Early VLE rows: 2139170
Minimum date: 0
Maximum date: 30


In [97]:
early_engagement = (
    early_vle
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation"
        ]
    )
    .agg(
        clicks_first_30_days=("sum_click", "sum"),
        active_days_first_30=("date", "nunique")
    )
    .reset_index()
)

In [98]:
early_engagement.head()

,id_student,code_module,code_presentation,clicks_first_30_days,active_days_first_30
0,6516,AAA,2014J,571,21
1,8462,DDD,2013J,325,17
2,8462,DDD,2014J,9,1
3,11391,AAA,2013J,326,9
4,23629,BBB,2013B,37,5


In [99]:
early_engagement["avg_clicks_first_30_active_day"] = np.where(
    early_engagement["active_days_first_30"] > 0,
    early_engagement["clicks_first_30_days"] /
    early_engagement["active_days_first_30"],
    0
)

In [100]:
early_engagement[
    [
        "clicks_first_30_days",
        "active_days_first_30",
        "avg_clicks_first_30_active_day"
    ]
].describe()

,clicks_first_30_days,active_days_first_30,avg_clicks_first_30_active_day
count,27971.000000,27971.000000,27971.000000
mean,291.610847,11.858282,21.254310
std,345.904371,7.709422,15.063573
min,1.000000,1.000000,1.000000
25%,74.000000,6.000000,11.066667
50%,184.000000,11.000000,17.875000
75%,388.000000,17.000000,27.707108
max,6495.000000,31.000000,260.368421


In [101]:
print(
    "Duplicate student-course-presentation records:",
    early_engagement.duplicated(
        subset=[
            "id_student",
            "code_module",
            "code_presentation"
        ]
    ).sum()
)

Duplicate student-course-presentation records: 0


In [102]:
vle_features = vle_engagement.merge(
    early_engagement,
    on=[
        "id_student",
        "code_module",
        "code_presentation"
    ],
    how="left",
    validate="one_to_one"
)

In [103]:
early_columns = [
    "clicks_first_30_days",
    "active_days_first_30",
    "avg_clicks_first_30_active_day"
]

vle_features[early_columns] = (
    vle_features[early_columns]
    .fillna(0)
)

In [104]:
print("Rows:", len(vle_features))
print("Columns:", len(vle_features.columns))

Rows: 29228
Columns: 15


In [105]:
print(
    "Duplicate records:",
    vle_features.duplicated(
        subset=[
            "id_student",
            "code_module",
            "code_presentation"
        ]
    ).sum()
)

Duplicate records: 0


In [106]:
vle_features[early_columns].isnull().sum()

clicks_first_30_days              0
active_days_first_30              0
avg_clicks_first_30_active_day    0
dtype: int64

In [107]:
early_vle.shape

(2139170, 6)

In [108]:
early_vle["date"].min(), early_vle["date"].max()

(np.int64(0), np.int64(30))

In [109]:
early_engagement.shape

(27971, 6)

In [110]:
early_engagement.duplicated(
    subset=[
        "id_student",
        "code_module",
        "code_presentation"
    ]
).sum()

np.int64(0)

In [111]:
vle_features.shape

(29228, 15)

In [112]:
vle_features[early_columns].isnull().sum()

clicks_first_30_days              0
active_days_first_30              0
avg_clicks_first_30_active_day    0
dtype: int64

## Step 21 — Assessment Features

Assessment information is combined with student assessment submissions to create learner-level academic performance features.

The features will be aggregated at:

Student + Course + Course Presentation

In [113]:
print("Assessments columns:")
print(assessments.columns.tolist())

print("\nStudent Assessment columns:")
print(student_assessment.columns.tolist())

Assessments columns:
['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight']

Student Assessment columns:
['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']


In [114]:
assessments.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0


In [115]:
student_assessment.head()

,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0


In [117]:

student_assessment.columns.tolist()

['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']

In [118]:
assessments.columns.tolist()

['code_module',
 'code_presentation',
 'id_assessment',
 'assessment_type',
 'date',
 'weight']

In [119]:
assessments.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0


## Step 22 — Combine Assessment Metadata and Student Results

The student assessment records contain student-level submission and score information, while the assessments table contains course, presentation, assessment type, date, and weight information.

The two datasets are joined using `id_assessment` so that each student score can be associated with the correct course and presentation.

In [120]:
student_assessment_enriched = student_assessment.merge(
    assessments[
        [
            "id_assessment",
            "code_module",
            "code_presentation",
            "assessment_type",
            "date",
            "weight"
        ]
    ],
    on="id_assessment",
    how="left",
    validate="many_to_one"
)

In [121]:
print("Rows:", len(student_assessment_enriched))
print("Columns:", len(student_assessment_enriched.columns))

Rows: 173912
Columns: 10


In [122]:
student_assessment_enriched.isnull().sum()

id_assessment           0
id_student              0
date_submitted          0
is_banked               0
score                 173
code_module             0
code_presentation       0
assessment_type         0
date                 2865
weight                  0
dtype: int64

In [123]:
student_assessment_enriched.head()

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0


In [124]:
print(
    "Student assessment IDs not found in assessments:",
    len(
        set(student_assessment["id_assessment"])
        - set(assessments["id_assessment"])
    )
)

Student assessment IDs not found in assessments: 0


## Step 23 — Assessment Score Validation

Assessment scores are examined to identify invalid values before creating student-level performance features.

In [125]:
student_assessment_enriched["score"].describe()

count    173739.000000
mean         75.799573
std          18.798107
min           0.000000
25%          65.000000
50%          80.000000
75%          90.000000
max         100.000000
Name: score, dtype: float64

In [126]:
invalid_scores = student_assessment_enriched[
    (student_assessment_enriched["score"] < 0) |
    (student_assessment_enriched["score"] > 100)
]

invalid_scores.shape

(0, 10)

In [127]:
student_assessment_enriched["score"].isnull().sum()

np.int64(173)

In [128]:
assessment_features = (
    student_assessment_enriched
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation"
        ]
    )
    .agg(
        assessment_count=("id_assessment", "nunique"),
        average_score=("score", "mean"),
        median_score=("score", "median"),
        maximum_score=("score", "max"),
        minimum_score=("score", "min"),
        assessment_active_days=("date_submitted", "nunique")
    )
    .reset_index()
)

In [129]:
available_assessments = (
    assessments
    .groupby(
        [
            "code_module",
            "code_presentation"
        ]
    )["id_assessment"]
    .nunique()
    .reset_index(name="available_assessments")
)

In [130]:
assessment_features = assessment_features.merge(
    available_assessments,
    on=[
        "code_module",
        "code_presentation"
    ],
    how="left",
    validate="many_to_one"
)

In [131]:
assessment_features["assessment_participation_rate"] = (
    assessment_features["assessment_count"] /
    assessment_features["available_assessments"]
)

In [132]:
assessment_features[
    [
        "assessment_count",
        "available_assessments",
        "assessment_participation_rate"
    ]
].describe()

,assessment_count,available_assessments,assessment_participation_rate
count,25843.000000,25843.000000,25843.000000
mean,6.729559,9.945478,0.681323
std,3.771381,2.938066,0.301592
min,1.000000,5.000000,0.071429
25%,4.000000,7.000000,0.416667
50%,7.000000,10.000000,0.833333
75%,10.000000,13.000000,0.916667
max,14.000000,14.000000,1.000000


In [133]:
assessment_features[
    (assessment_features["assessment_participation_rate"] < 0) |
    (assessment_features["assessment_participation_rate"] > 1)
].shape

(0, 11)

In [134]:
assessment_features.duplicated(
    subset=[
        "id_student",
        "code_module",
        "code_presentation"
    ]
).sum()

np.int64(0)

In [135]:
student_assessment_enriched["weighted_score_component"] = (
    student_assessment_enriched["score"] *
    student_assessment_enriched["weight"]
)

In [136]:
weighted_assessment = (
    student_assessment_enriched
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation"
        ]
    )
    .agg(
        weighted_score_sum=("weighted_score_component", "sum"),
        submitted_weight=("weight", "sum")
    )
    .reset_index()
)

In [137]:
weighted_assessment["weighted_average_score"] = (
    weighted_assessment["weighted_score_sum"] /
    weighted_assessment["submitted_weight"]
)

In [138]:
assessment_features = assessment_features.merge(
    weighted_assessment[
        [
            "id_student",
            "code_module",
            "code_presentation",
            "weighted_average_score"
        ]
    ],
    on=[
        "id_student",
        "code_module",
        "code_presentation"
    ],
    how="left",
    validate="one_to_one"
)

In [139]:
assessment_features["weighted_average_score"].describe()

count    23506.000000
mean        69.526300
std         16.961600
min          0.000000
25%         59.666667
50%         72.000000
75%         82.000000
max        100.000000
Name: weighted_average_score, dtype: float64

In [140]:
assessment_features[
    (assessment_features["weighted_average_score"] < 0) |
    (assessment_features["weighted_average_score"] > 100)
].shape

(0, 12)

In [141]:
print("Rows:", len(assessment_features))
print("Columns:", len(assessment_features.columns))

Rows: 25843
Columns: 12


In [142]:
print(
    "Duplicate student-course-presentation records:",
    assessment_features.duplicated(
        subset=[
            "id_student",
            "code_module",
            "code_presentation"
        ]
    ).sum()
)

Duplicate student-course-presentation records: 0


In [143]:
assessment_features.head()

,id_student,code_module,code_presentation,assessment_count,average_score,median_score,maximum_score,minimum_score,assessment_active_days,available_assessments,assessment_participation_rate,weighted_average_score
0,6516,AAA,2014J,5,61.800000,61.0,77.0,48.0,5,6,0.833333,63.50
1,8462,DDD,2013J,3,87.666667,87.0,93.0,83.0,3,7,0.428571,87.25
2,8462,DDD,2014J,4,86.500000,85.0,93.0,83.0,1,7,0.571429,86.00
3,11391,AAA,2013J,5,82.000000,82.0,85.0,78.0,5,6,0.833333,82.40
4,23629,BBB,2013B,4,82.500000,83.5,100.0,63.0,4,12,0.333333,66.76


In [144]:
assessment_features.isnull().sum()

id_student                          0
code_module                         0
code_presentation                   0
assessment_count                    0
average_score                      23
median_score                       23
maximum_score                      23
minimum_score                      23
assessment_active_days              0
available_assessments               0
assessment_participation_rate       0
weighted_average_score           2337
dtype: int64

In [145]:
student_assessment_enriched.shape

(173912, 11)

In [146]:
assessment_features.shape

(25843, 12)

In [147]:
assessment_features.isnull().sum()

id_student                          0
code_module                         0
code_presentation                   0
assessment_count                    0
average_score                      23
median_score                       23
maximum_score                      23
minimum_score                      23
assessment_active_days              0
available_assessments               0
assessment_participation_rate       0
weighted_average_score           2337
dtype: int64

In [148]:
assessment_features[
    [
        "assessment_count",
        "available_assessments",
        "assessment_participation_rate",
        "weighted_average_score"
    ]
].head()

,assessment_count,available_assessments,assessment_participation_rate,weighted_average_score
0,5,6,0.833333,63.50
1,3,7,0.428571,87.25
2,4,7,0.571429,86.00
3,5,6,0.833333,82.40
4,4,12,0.333333,66.76


In [149]:
assessment_features.duplicated(
    subset=[
        "id_student",
        "code_module",
        "code_presentation"
    ]
).sum()

np.int64(0)

In [150]:
assessment_features[
    assessment_features["weighted_average_score"].isna()
].head(20)

,id_student,code_module,code_presentation,assessment_count,average_score,median_score,maximum_score,minimum_score,assessment_active_days,available_assessments,assessment_participation_rate,weighted_average_score
7,24186,GGG,2014B,8,62.500000,60.0,80.0,40.0,5,10,0.800000,NaN
9,24391,GGG,2013J,9,88.888889,80.0,100.0,80.0,9,10,0.900000,NaN
16,26023,GGG,2013J,9,88.000000,100.0,100.0,53.0,9,10,0.900000,NaN
51,29985,GGG,2014B,9,71.888889,75.0,100.0,0.0,9,10,0.900000,NaN
58,31107,GGG,2014B,9,89.222222,100.0,100.0,72.0,9,10,0.900000,NaN
61,31205,GGG,2014J,4,90.500000,91.0,100.0,80.0,4,10,0.400000,NaN
69,32239,GGG,2013J,8,80.625000,90.0,100.0,55.0,8,10,0.800000,NaN
95,35001,GGG,2013J,2,75.000000,75.0,80.0,70.0,2,10,0.200000,NaN
130,41414,GGG,2014B,9,80.000000,80.0,100.0,40.0,8,10,0.900000,NaN
171,49123,GGG,2014B,9,85.111111,80.0,100.0,69.0,9,10,0.900000,NaN


In [151]:
weighted_assessment[
    weighted_assessment["weighted_average_score"].isna()
][
    [
        "id_student",
        "code_module",
        "code_presentation",
        "weighted_score_sum",
        "submitted_weight",
        "weighted_average_score"
    ]
].head(20)

,id_student,code_module,code_presentation,weighted_score_sum,submitted_weight,weighted_average_score
7,24186,GGG,2014B,0.0,0.0,NaN
9,24391,GGG,2013J,0.0,0.0,NaN
16,26023,GGG,2013J,0.0,0.0,NaN
51,29985,GGG,2014B,0.0,0.0,NaN
58,31107,GGG,2014B,0.0,0.0,NaN
61,31205,GGG,2014J,0.0,0.0,NaN
69,32239,GGG,2013J,0.0,0.0,NaN
95,35001,GGG,2013J,0.0,0.0,NaN
130,41414,GGG,2014B,0.0,0.0,NaN
171,49123,GGG,2014B,0.0,0.0,NaN


In [152]:
print("Missing submitted_weight:",
      weighted_assessment["submitted_weight"].isna().sum())

print("Zero submitted_weight:",
      (weighted_assessment["submitted_weight"] == 0).sum())

Missing submitted_weight: 0
Zero submitted_weight: 2337


In [153]:
student_assessment_enriched[
    student_assessment_enriched["score"].isna()
].shape

(173, 11)

## Weighted Assessment Score — Missing Value Treatment

Some student-course-presentation groups have a submitted assessment weight of zero.
For these groups, a weighted average score cannot be calculated because the denominator is zero.

These values are retained as missing rather than being replaced with zero, because zero would incorrectly imply a score of 0%.

In [154]:
assessment_features["weighted_average_score"].isna().sum()

np.int64(2337)

In [155]:
assessment_features["weighted_average_score"].isna().sum()

np.int64(2337)

In [156]:
master_df = master_df.merge(
    assessment_features,
    on=[
        "id_student",
        "code_module",
        "code_presentation"
    ],
    how="left",
    validate="one_to_one"
)

In [157]:
print("Rows:", len(master_df))
print("Columns:", len(master_df.columns))

Rows: 32593
Columns: 27


In [158]:
master_df[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "final_result",
        "dropout",
        "assessment_count",
        "average_score",
        "assessment_participation_rate",
        "weighted_average_score"
    ]
].head()

,id_student,code_module,code_presentation,final_result,dropout,assessment_count,average_score,assessment_participation_rate,weighted_average_score
0,11391,AAA,2013J,Pass,0,5.0,82.0,0.833333,82.4
1,28400,AAA,2013J,Pass,0,5.0,66.4,0.833333,65.4
2,30268,AAA,2013J,Withdrawn,1,NaN,NaN,NaN,NaN
3,31604,AAA,2013J,Pass,0,5.0,76.0,0.833333,76.3
4,32885,AAA,2013J,Pass,0,5.0,54.4,0.833333,55.0


In [159]:
master_df[
    [
        "assessment_count",
        "assessment_participation_rate"
    ]
].isnull().sum()

assessment_count                 6750
assessment_participation_rate    6750
dtype: int64

In [160]:
assessment_count_columns = [
    "assessment_count",
    "assessment_participation_rate"
]

master_df[assessment_count_columns] = (
    master_df[assessment_count_columns]
    .fillna(0)
)

In [161]:
master_df[
    [
        "assessment_count",
        "assessment_participation_rate"
    ]
].isnull().sum()

assessment_count                 0
assessment_participation_rate    0
dtype: int64

In [162]:
master_df[
    [
        "average_score",
        "median_score",
        "minimum_score",
        "maximum_score",
        "weighted_average_score"
    ]
].isnull().sum()

average_score             6773
median_score              6773
minimum_score             6773
maximum_score             6773
weighted_average_score    9087
dtype: int64

In [163]:
master_df = master_df.merge(
    vle_features,
    on=[
        "id_student",
        "code_module",
        "code_presentation"
    ],
    how="left",
    validate="one_to_one"
)

In [164]:
print("Rows:", len(master_df))
print("Columns:", len(master_df.columns))

Rows: 32593
Columns: 39


In [166]:
master_df[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "total_clicks",
        "active_days",
        "engagement_coverage",
        "clicks_first_30_days",
        "active_days_first_30"
    ]
].head()

,id_student,code_module,code_presentation,total_clicks,active_days,engagement_coverage,clicks_first_30_days,active_days_first_30
0,11391,AAA,2013J,922.0,40.0,0.145522,326.0,9.0
1,28400,AAA,2013J,1409.0,80.0,0.272388,397.0,12.0
2,30268,AAA,2013J,260.0,12.0,0.022388,159.0,6.0
3,31604,AAA,2013J,2007.0,123.0,0.440299,345.0,19.0
4,32885,AAA,2013J,1012.0,70.0,0.231343,267.0,16.0


In [167]:
master_df[
    [
        "total_clicks",
        "active_days",
        "engagement_coverage",
        "clicks_first_30_days",
        "active_days_first_30"
    ]
].isnull().sum()

total_clicks            3365
active_days             3365
engagement_coverage     3365
clicks_first_30_days    3365
active_days_first_30    3365
dtype: int64

In [168]:
print(
    "Duplicate master records:",
    master_df.duplicated(
        subset=[
            "id_student",
            "code_module",
            "code_presentation"
        ]
    ).sum()
)

Duplicate master records: 0


In [169]:
vle_engagement_columns = [
    "total_clicks",
    "active_days",
    "engagement_coverage",
    "clicks_first_30_days",
    "active_days_first_30",
    "avg_clicks_per_active_day",
    "avg_clicks_first_30_active_day"
]

master_df[vle_engagement_columns] = (
    master_df[vle_engagement_columns]
    .fillna(0)
)

In [170]:
master_df[vle_engagement_columns].isnull().sum()

total_clicks                      0
active_days                       0
engagement_coverage               0
clicks_first_30_days              0
active_days_first_30              0
avg_clicks_per_active_day         0
avg_clicks_first_30_active_day    0
dtype: int64

In [171]:
print("Rows:", len(master_df))
print(
    "Duplicate master records:",
    master_df.duplicated(
        subset=[
            "id_student",
            "code_module",
            "code_presentation"
        ]
    ).sum()
)

Rows: 32593
Duplicate master records: 0


In [172]:
print("Master dataset shape:", master_df.shape)

Master dataset shape: (32593, 39)


In [173]:
master_df.columns.tolist()

['code_module',
 'code_presentation',
 'id_student',
 'gender',
 'region',
 'highest_education',
 'imd_band',
 'age_band',
 'num_of_prev_attempts',
 'studied_credits',
 'disability',
 'final_result',
 'dropout',
 'date_registration',
 'date_unregistration',
 'registration_missing',
 'was_unregistered',
 'days_until_unregistration',
 'assessment_count',
 'average_score',
 'median_score',
 'maximum_score',
 'minimum_score',
 'assessment_active_days',
 'available_assessments',
 'assessment_participation_rate',
 'weighted_average_score',
 'total_clicks',
 'active_days',
 'first_activity_date',
 'last_activity_date',
 'activity_span_days',
 'avg_clicks_per_active_day',
 'module_presentation_length',
 'engagement_coverage',
 'active_days_in_course',
 'clicks_first_30_days',
 'active_days_first_30',
 'avg_clicks_first_30_active_day']

In [174]:
master_df.duplicated(
    subset=[
        "id_student",
        "code_module",
        "code_presentation"
    ]
).sum()

np.int64(0)

In [175]:
master_df["dropout"].value_counts(dropna=False)

dropout
0    22437
1    10156
Name: count, dtype: int64

In [176]:
master_df.isnull().sum()

code_module                           0
code_presentation                     0
id_student                            0
gender                                0
region                                0
highest_education                     0
imd_band                              0
age_band                              0
num_of_prev_attempts                  0
studied_credits                       0
disability                            0
final_result                          0
dropout                               0
date_registration                    45
date_unregistration               22521
registration_missing                  0
was_unregistered                      0
days_until_unregistration         22560
assessment_count                      0
average_score                      6773
median_score                       6773
maximum_score                      6773
minimum_score                      6773
assessment_active_days             6750
available_assessments              6750


In [177]:
master_df[
    [
        "total_clicks",
        "active_days",
        "engagement_coverage",
        "clicks_first_30_days",
        "active_days_first_30",
        "assessment_participation_rate",
        "average_score",
        "weighted_average_score"
    ]
].describe()

,total_clicks,active_days,engagement_coverage,clicks_first_30_days,active_days_first_30,assessment_participation_rate,average_score,weighted_average_score
count,32593.000000,32593.000000,32593.000000,32593.000000,32593.000000,32593.000000,25820.000000,23506.000000
mean,1176.420182,55.475685,0.203598,250.257632,10.176664,0.540221,72.768254,69.526300
std,1649.137941,54.515290,0.203433,336.200973,8.253498,0.385160,16.374997,16.961600
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,134.000000,11.000000,0.033457,32.000000,3.000000,0.100000,64.666667,59.666667
50%,575.000000,40.000000,0.144981,143.000000,9.000000,0.714286,76.000000,72.000000
75%,1527.000000,85.000000,0.317164,343.000000,16.000000,0.900000,84.545455,82.000000
max,23513.000000,286.000000,1.000000,6495.000000,31.000000,1.000000,100.000000,100.000000


In [178]:
master_df.to_csv(
    "../data/processed/master_analytical_dataset.csv",
    index=False
)

In [179]:
print(
    os.path.exists(
        "../data/processed/master_analytical_dataset.csv"
    )
)

True
